In [ ]:
!pip install torchaudio datasets transformers jiwer pythainlp evaluate deepcut -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.3/19.3 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 83.9 MB/s eta 0:00:00


In [ ]:
folder_url = 'Your-dataset-folder-url-here'
!gdown {folder_url}

Downloading...
From (original): https://drive.google.com/uc?id=1ZVKX6YoYuc9zMpHy48WF-dUuE6Otz-Ks
From (redirected): https://drive.google.com/uc?id=1ZVKX6YoYuc9zMpHy48WF-dUuE6Otz-Ks&confirm=t&uuid=7bc7575b-bf48-4c1e-9758-f1eca38252f5
To: /content/cv-corpus-7.0-2021-07-21-th.tar.gz
100% 5.37G/5.37G [01:34<00:00, 56.7MB/s]


In [ ]:
!tar -xvzf /content/cv-corpus-7.0-2021-07-21-th.tar.gz -C /content/

เอาต์พุตของการสตรีมมีการตัดเหลือเพียง 5000 บรรทัดสุดท้าย
cv-corpus-7.0-2021-07-21/th/clips/common_voice_th_27269603.mp3
cv-corpus-7.0-2021-07-21/th/clips/common_voice_th_27269604.mp3
cv-corpus-7.0-2021-07-21/th/clips/common_voice_th_27269606.mp3
cv-corpus-7.0-2021-07-21/th/clips/common_voice_th_27269608.mp3
cv-corpus-7.0-2021-07-21/th/clips/common_voice_th_27269609.mp3
cv-corpus-7.0-2021-07-21/th/clips/common_voice_th_27269611.mp3
cv-corpus-7.0-2021-07-21/th/clips/common_voice_th_27269613.mp3
cv-corpus-7.0-2021-07-21/th/clips/common_voice_th_27269615.mp3
cv-corpus-7.0-2021-07-21/th/clips/common_voice_th_27269616.mp3
cv-corpus-7.0-2021-07-21/th/clips/common_voice_th_27269617.mp3
cv-corpus-7.0-2021-07-21/th/clips/common_voice_th_27269618.mp3
cv-corpus-7.0-2021-07-21/th/clips/common_voice_th_27269619.mp3
cv-corpus-7.0-2021-07-21/th/clips/common_voice_th_27269682.mp3
cv-corpus-7.0-2021-07-21/th/clips/common_voice_th_27269683.mp3
cv-corpus-7.0-2021-07-21/th/clips/common_voice_th_27269684.mp

In [ ]:
import torch
import torchaudio
from datasets import Dataset
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
from pythainlp.tokenize import word_tokenize
import re
import evaluate
import os

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive/')

Mounted at /content/gdrive/


In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
test_dataset = Dataset.from_csv('Your-dataset-folder-url-here', delimiter='\t')

wer = evaluate.load("wer")
cer = evaluate.load("cer")

processor = Wav2Vec2Processor.from_pretrained("Your-model-folder-url-here")
model = Wav2Vec2ForCTC.from_pretrained("Your-model-folder-url-here")
model.to("cuda")

chars_to_ignore_regex = '[\,\?\.\!\-\;\:\"\“]'
resampler = torchaudio.transforms.Resample(48_000, 16_000)

## For Thai NLP Library, please feel free to check https://pythainlp.github.io/docs/2.2/api/tokenize.html
def th_tokenize(batch):
    batch["sentence"] = " ".join(word_tokenize(batch["sentence"], engine="deepcut"))
    # batch["sentence"] = " ".join(word_tokenize(batch["sentence"], engine="attacut"))
    return batch

# Preprocessing the datasets.
# # We need to read the audio files as arrays
def speech_file_to_array_fn(batch):
    audio_file_path = os.path.join("Your-audio-files-folder-url-here", batch["path"])

    if not os.path.exists(audio_file_path):
        print(f"File does not exist: {audio_file_path}")
        return batch

    batch["sentence"] = re.sub(chars_to_ignore_regex, '', batch["sentence"]).lower()
    speech_array, sampling_rate = torchaudio.load(audio_file_path)
    batch["speech"] = resampler(speech_array).squeeze().numpy()
    return batch

In [ ]:
model.freeze_feature_extractor()

In [ ]:
# test = test_dataset.map(th_tokenize).map(speech_file_to_array_fn)
test = test_dataset.map(speech_file_to_array_fn)

Map:   0%|          | 0/4276 [00:00<?, ? examples/s]

In [ ]:
# Preprocessing the datasets.
# We need to read the aduio files as arrays
def predict(batch):
    inputs = processor(batch["speech"], sampling_rate=16_000, return_tensors="pt", padding=True)

    with torch.no_grad():
        logits = model(inputs.input_values.to("cuda"), attention_mask=inputs.attention_mask.to("cuda")).logits

    pred_ids = torch.argmax(logits, dim=-1)
    batch["pred_strings"] = processor.batch_decode(pred_ids)
    return batch

# **Evaluate Model**

In [ ]:
result = test_dataset.map(predict, batched=True, batch_size=64, cache_file_name=None)

print("WER: {:2f}".format(100 * wer.compute(predictions=result["pred_strings"], references=result["sentence"])))
print("CER: {:2f}".format(100 * cer.compute(predictions=result["pred_strings"], references=result["sentence"])))

Parameter 'function'=<function predict at 0x7e08b537c360> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/4276 [00:00<?, ? examples/s]

WER: 22.269664
CER: 7.394771


In [ ]:
print("ref | pred")
for obj in zip(test_dataset["sentence"][:15], result["pred_strings"]):
  print(obj)

ref | pred
('ในเดือนกุมภาพันธ์ มันจะเป็นวันครบรอบของเรา', 'ใน เดือน กลุ่มพาพันธุ์ มัน จะ เป็น วัน ครบรอก ของ เรา')
('เป้าหมายของเราก็ไม่ถูกต้อง', 'เปาหมาย ของ เรา ก็ ไม่ ถูกต้อง เ')
('เธอจะทำทุกอย่างเท่าที่ทำได้เพื่อทำให้ฉันมีความสุข', 'เธอ จะ ทำ ทุก อย่าง เท่า ที่ ทำ ได้ เพื่อ ทำ ให้ ฉัน มี ความ สุข')
('เขาจัดเตรียมถุงนอนของเขาเหมือนกับกระสอบ และเขาก็มุ่งหน้าไปยังข้างคลอง', 'เขา จัด เตรียม ถุง นอน ของ เขา หมือน กับ กระสอบ และ เขา ก็ มุ่งหน้า ไป ยัง ห้าคลอง')
('อ้วกครับ แสงสีมาพร้อม', 'อ้วคับ แสง สี ม้า พร้อม ััับบ')
('การโจมตีที่เซิร์ฟเวอร์รูทของพวกเราทำให้ผู้ดูแลทำงานหนักเกินไป', 'การ โจมตี ที่ เซิร์ฟเวอร์ รู้ด ของ พวก เรา ทำ ให้ ผู้ ดูแล ทำ งาน หนัก เกิน ไป')
('ตีแสกหน้า', 'ตี ใสก หน้า')
('ค่าใช้จ่ายส่วนใหญ่ยังเกี่ยวกับชีวิตส่วนตัว', 'ค่า ใจ จาด ส่วน ใหญ่ ยังเกี่ยว กับ ชีวิต ส่วน ตัว')
('มันทำให้ฉันคิดถึง', 'มัน ทำ ให้ ฉัน คิด ถึง  ำ')
('สันติภาพจงมีแด่ท่านและพระคุณต่อพระพักตร์พระเจ้า', 'สันติภาพ จง มี แด่ ท่าน และ พระคุณ ต่อ พระพัรคพระเจ้า')
('พัดลมตั้งโต๊ะ', 'พัด ลม ตั้ง โต๊ะ')
('

# **Temp1**

In [ ]:
result = test_dataset.map(predict, batched=True, batch_size=64, cache_file_name=None)

print("WER: {:2f}".format(100 * wer.compute(predictions=result["pred_strings"], references=result["sentence"])))
print("CER: {:2f}".format(100 * cer.compute(predictions=result["pred_strings"], references=result["sentence"])))

Map:   0%|          | 0/9712 [00:00<?, ? examples/s]

WER: 26.871526
CER: 9.969179


In [ ]:
print("ref | pred")
for obj in zip(result["sentence"][:15], result["pred_strings"]):
  print(obj)

ref | pred
('ใคร เป็น ผู้ รับ ', 'ใคร เป็น ผู้ รับ')
('ผู้ ชาย คือ ช้างเท้า หน้า   แต่ ผู้ หญิง คือ ควาญ ช้าง ', 'ผู้ ชาย คือ ชาง ซาว หน้า แต่ ผู้ หญิง คือ ขวาม ช้างแอแฮะ')
('ไม่ ต้อง สงสัย เลย   เธอ เป็น คน เขียน จดหมาย นี้ ', 'ไม่ ต้อง สงสั เลย เธอ เป็น คน เขียน จดหมาย นี้')
('คุณ คือ คน ที่ ดี ที่สุด   คุณ ผู้ หญิง   เขา พูด ด้วย ท่าที สุภาพ ', 'คุณ คือ คน ที่ ดี ที่สุด คุณ ผื่อ หญิง เขา พูด ด้วย ท่าที สุภาพ')
('ลอง ชิม สลัด นี้ ดู สิ ', 'ลอง ชิมสลัส หนี่ ดู สิก')
('ทุเรียน เป็น ผลไม้ ขึ้น ชื่อ ของ ไทย ', 'ทูเลียน เป็น บวน ลำ ไม้ ขึ้น ชื่อ ของ ไทย ีี')
('ไม่ ต้อง สงสัย เลย ว่า เขา กำลัง หา โทรเลข อยู่ ', 'ไม่ ต้อง สงสัย ใน ว่า เขา กำลัง หา ทรเลข อยู่')
('แซ็ก อยาก เป็น นัก เก็ต ', 'แซ็ก ยาก เป็น นัเกต')
('ยี่ สิบ ห้า ', 'ยี่สิบ ห้า ่่่่่่่่่')
('หลัง จาก คิด พิจารณา   เขา ก็ ชี้แจง ประเด็น ให้ ภรรยา ของ เขา ', 'ลัง จาก คิด พิจารณา เคา ก็ ชี้แจง ประเด็น ให้ ประยา ของ เขา')
('ไป ก่อน ล่ะ ', 'ไป ก่อน ละ')
('เขา นั่ง ที่ โต๊ะ ', 'เขา นั่ง ที่ โต๊ะ งง')
('ผู้ มี มนุษยธรรม รัก แนวทาง ของ 

# **Temp2**

In [ ]:
result = test.map(predict, batched=True, batch_size=64, cache_file_name=None)

print("WER: {:2f}".format(100 * wer.compute(predictions=result["pred_strings"], references=result["sentence"])))
print("CER: {:2f}".format(100 * cer.compute(predictions=result["pred_strings"], references=result["sentence"])))

Parameter 'function'=<function predict at 0x7f5e44201f80> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/4276 [00:00<?, ? examples/s]

WER: 19.565552
CER: 6.348251


In [ ]:
print("ref | pred")
for obj in zip(test_dataset["sentence"][:15], result["pred_strings"]):
  print(obj)

ref | pred
('ใน เดือน กุมภาพันธ์   มัน จะ เป็น วัน ครบ รอบ ของ เรา ', 'ใน เดีืยน กุมพาพันธ์ มัน จะ เป็น วัน ครอบ ๆ ของ เรา')
('เป้าหมาย ของ เรา ก็ ไม่ ถูกต้อง ', 'เปา หมาย ของ เรา ก็ ไม่ ถูกต้อง ห')
('เธอ จะ ทำ ทุก อย่าง เท่า ที่ ทำ ได้ เพื่อ ทำ ให้ ฉัน มี ความ สุข ', 'เธอ จะ ทำ ทุกอย่าง เท่า ที่ ทำ ได้ เพื่อ ทำ ให้ ฉัน มี ความ สุข')
('เขา จัด เตรียม ถุง นอน ของ เขา เหมือน กับ กระสอบ   และ เขา ก็ มุ่งหน้า ไป ยัง ข้าง คลอง ', 'เขา จัด เตรียม ถุง นอน ของ เขา เหมือน กับ กระสอบ และ เขา ก็ มุ่งหน้า ไป ยัง ข่างคลอง')
('อ้วก ครับ   แสง สี มา พร้อม ', 'อ้วครับ แสง สี ม้า พร้อม')
('การ โจมตี ที่ เซิร์ฟเวอร์ รูท ของ พวก เรา ทำ ให้ ผู้ ดูแล ทำ งาน หนัก เกิน ไป ', 'การ โจมตี ที่ เซิร์ฟเวอร์รู้ด ของ พวก เรา ทำ ให้ ผู้ ดูแล ทำ งาน หนัก เกิน ไป')
('ตี แสก หน้า ', 'ตี ใสก หน้า')
('ค่า ใช้จ่าย ส่วน ใหญ่ ยัง เกี่ยว กับ ชีวิต ส่วน ตัว ', 'ค้า ใจ จาก  ส่วน ใหญ่ ยัง เกี่ยว กับ ชีวิต ส่วน ตัว')
('มัน ทำ ให้ ฉัน คิด ถึง ', 'มัน ทำ ให้ ฉัน คิด ถึง')
('สันติภาพ จง มี แด่ ท่าน และ พระคุณ ต่อ พระพักตร์ พระเจ้า '